# Microsoft Agent Framework

# Introducción a Microsoft Agent Framework

## ¿Qué es Microsoft Agent Framework?

**Microsoft Agent Framework (MAF)** es un framework de código abierto desarrollado por Microsoft para construir aplicaciones de inteligencia artificial basadas en agentes y flujos de trabajo (*workflows*) utilizando Python y .NET.

Proporciona componentes para conectar modelos de lenguaje (LLM), definir agentes, integrar herramientas y coordinar la ejecución de tareas. Su objetivo es facilitar el desarrollo de aplicaciones que combinan las capacidades de los modelos de lenguaje con código, servicios externos y procesos definidos por el desarrollador.

A diferencia de una aplicación que únicamente envía una consulta a un modelo de lenguaje y recibe una respuesta, un agente puede utilizar herramientas disponibles para obtener información o ejecutar operaciones, y emplear sus resultados para resolver una solicitud. Las acciones que puede realizar dependen de las herramientas, los permisos y las reglas que configure la aplicación.

Microsoft Agent Framework es el sucesor de las iniciativas AutoGen y Semantic Kernel, y reúne conceptos de agentes y orquestación con capacidades orientadas al desarrollo de aplicaciones robustas y mantenibles.

## Características principales

- **Integración con modelos de lenguaje:** permite conectar agentes con distintos proveedores de modelos, incluidos Azure OpenAI, OpenAI y Microsoft Foundry, mediante las integraciones correspondientes.

- **Uso de herramientas (*tools*):** permite exponer funciones y servicios para que los agentes puedan realizar cálculos, consultar información o ejecutar operaciones autorizadas.

- **Gestión del estado conversacional:** proporciona mecanismos para mantener el estado y el contexto de las interacciones entre distintos turnos. La persistencia y la memoria de largo plazo dependen de los componentes y la configuración utilizados.

- **Orquestación de workflows:** permite definir flujos de ejecución que coordinan agentes y funciones mediante pasos secuenciales, concurrentes y otros patrones de control.

- **Sistemas multiagente:** facilita la coordinación de agentes especializados mediante patrones de colaboración, transferencia de tareas y ejecución organizada.

- **Integración con sistemas externos:** admite mecanismos como el *Model Context Protocol* (MCP) para conectar agentes con herramientas y recursos externos compatibles.

- **Middleware y extensibilidad:** permite incorporar lógica personalizada para interceptar determinadas operaciones, gestionar errores y aplicar controles adicionales.

- **Observabilidad:** ofrece instrumentación compatible con OpenTelemetry para analizar ejecuciones, obtener trazas y métricas, y facilitar la supervisión y depuración de las aplicaciones.

- **Control de ejecución e intervención humana:** sus capacidades de workflows permiten implementar procesos con puntos de aprobación humana y mecanismos de gestión del estado, según el diseño y los componentes utilizados.

En conjunto, Microsoft Agent Framework proporciona una base de desarrollo para crear aplicaciones de IA que combinan modelos de lenguaje, herramientas y lógica de negocio. El framework ofrece los componentes de construcción y orquestación, mientras que la aplicación define las reglas, los permisos, las integraciones y los controles necesarios para cada caso de uso.

**Documentación oficial:** https://learn.microsoft.com/en-us/agent-framework/  
**Repositorio oficial:** https://github.com/microsoft/agent-framework

In [2]:

import os
from dotenv import load_dotenv
from agent_framework import Agent, AgentExecutor
from agent_framework import WorkflowBuilder, Executor, WorkflowContext, handler
from agent_framework.orchestrations import SequentialBuilder, ConcurrentBuilder
from typing import Never
from pydantic import BaseModel
from agent_framework.openai import OpenAIChatClient
from azure.identity import DefaultAzureCredential

load_dotenv()
endpoint_llm = os.getenv("FOUNDRY_MODEL_ENDPOINT")
model_llm = os.getenv("FOUNDRY_MODEL_DEPLOYMENT")
version_llm = os.getenv("FOUNDRY_MODEL_VERSION")


In [3]:
# Credenciales
credential = DefaultAzureCredential()

# Consulta a un LLM utilizndo Agent Framework
client = OpenAIChatClient(
    model=model_llm,  # Nombre exacto del deployment
    base_url=endpoint_llm,
    credential=credential,
)

agent = Agent(
    client=client,
    instructions="Responde de forma clara y en español.",
)

respuesta = await agent.run(
    "Explica brevemente qué es la inteligencia artificial generativa."
)

print(respuesta.text)

La inteligencia artificial generativa es una rama de la IA capaz de crear contenido nuevo —como texto, imágenes, música, videos o código— a partir de patrones aprendidos de grandes cantidades de datos. Por ejemplo, puede redactar un texto a partir de una instrucción o generar una imagen según una descripción.


In [4]:
# Se ejecuta un workflow con Agent Framework
# Paso 1: transformar el texto
class ConvertirMayusculas(Executor):
    """Metodo para convertir el texto en mayusculas"""
    @handler # Decorador para declarar el metodo como manejador de mensajes
    async def procesar(self,texto: str,ctx: WorkflowContext[str]) -> None:
        resultado = texto.upper()
        await ctx.send_message(resultado)


# Paso 2: preparar el resultado final
class AgregarMensaje(Executor):
    """Metodo para agregar texto al inicio de la respuesta"""
    @handler # Decorador para declarar el metodo como manejador de mensajes
    async def procesar(self,texto: str, ctx: WorkflowContext[Never, str]) -> None:
        # Input:
        #   str: Texto que recibe en la consulta
        #   ctx: Manejador de mensajes a lo largo de todo el workflow
        resultado = f"Resultado del workflow: {texto}"
        await ctx.yield_output(resultado)


async def main():
    # Se definen los pasos que se van a ejecutar
    paso1 = ConvertirMayusculas(id="convertir") # El id es un identificador unico del paso de ejecucion
    paso2 = AgregarMensaje(id="agregar_mensaje")

    # Se configura el workflow
    workflow = (
        WorkflowBuilder(start_executor=paso1) # Se define el paso inicial
        .add_edge(paso1, paso2) # Se agrega una arista entre el primer paso y el segundo paso
        .build() # Se construye el flujo de ejecucion
    )

    # Se ejecuta el flujo de trabajo
    resultado = await workflow.run("hola mundo")
    print(resultado.get_outputs())


await main()

['Resultado del workflow: HOLA MUNDO']


In [5]:
# Workflows con agente
# Ejecutor 1: deterministico
class Ejecutor1(Executor):
    """Metodo para indicar que la generacion debe ser en idioma ingles"""
    @handler # Decorador para declarar el metodo como manejador de mensajes
    async def procesar(self,texto: str,ctx: WorkflowContext[str]) -> None:
        prompt = f"""
        Responde en inglés a la siguiente pregunta.

        Pregunta del usuario:
        {texto}
        """
        await ctx.send_message(prompt)

# Agente de IA: genera el contenido
agente = Agent(
    client=client,
    name="AgenteExplicador",
    instructions=(
        "Explica conceptos de inteligencia artificial "
        "de forma clara, precisa y concisa."
    ),
)

# Ejecutor que invoca al agente
class ConsultarAgente(Executor):
    def __init__(self, agente: Agent, id: str):
        super().__init__(id=id)
        self.agente = agente # Instancia de Agent

    @handler # Decorador para indicar al workflow que es un metodo manejador de mensajes
    async def procesar(self,pregunta: str,ctx: WorkflowContext[Never, str],) -> None:
        respuesta = await self.agente.run(pregunta)
        await ctx.yield_output(respuesta.text)


async def main():
    paso1 = Ejecutor1(id="ejecutor")
    paso_agente = ConsultarAgente(agente, id="consultar_agente")

    workflow = (
        WorkflowBuilder(start_executor=paso1)
        .add_edge(paso1,paso_agente)
        .build()
    )

    resultado = await workflow.run(
        "¿Qué es la IA generativa?"
    )

    print(resultado.get_outputs())


await main()

['Generative AI is a type of artificial intelligence that can create new content—such as text, images, music, video, or code—by learning patterns from existing data. Examples include chatbots that write responses, tools that generate images from descriptions, and systems that compose music or produce software code.']


In [6]:
# Workflow con multiples agentes
redactor = Agent(
    client=client,
    name="Redactor",
    instructions=(
        "Eres un redactor técnico. "
        "Redacta una explicación clara y precisa sobre el tema solicitado."
    ),
)

revisor = Agent(
    client=client,
    name="Revisor",
    instructions=(
        "Revisa la respuesta anterior. "
        "Corrige errores conceptuales, mejora la claridad y elimina "
        "redundancias. Devuelve la versión final mejorada."
    ),
)

# Ejecutor que invoca al agente
class RedactorExecutor(Executor):
    def __init__(self, agente: Agent, id: str):
        super().__init__(id=id)
        self.agente = agente # Instancia de Agent

    @handler # Decorador para indicar al workflow que es un metodo manejador de mensajes
    async def procesar(self,pregunta: str,ctx: WorkflowContext[str],) -> None:
        respuesta = await self.agente.run(pregunta)
        await ctx.send_message(respuesta.text) # Aca solo se pasa el mensaje al siguiente agente


class RevisorExecutor(Executor):
    def __init__(self, agente: Agent, id: str):
        super().__init__(id=id)
        self.agente = agente # Instancia de Agent

    @handler # Decorador para indicar al workflow que es un metodo manejador de mensajes
    async def procesar(self,texto: str,ctx: WorkflowContext[Never, str],) -> None:
        respuesta = await self.agente.run(f"Revisa y mejora el siguiente texto:\n\n{texto}")
        await ctx.yield_output(respuesta.text) # Este agente si responde

async def main():
    agente1 = RedactorExecutor(redactor, id="consultar_agente_redactor")
    agente2 = RevisorExecutor(revisor, id="consultar_agente_revisor")
    
    workflow = (
        WorkflowBuilder(start_executor=agente1)
        .add_edge(agente1, agente2)
        .build()
    )

    resultado = await workflow.run(
        "¿Qué es la IA generativa?"
    )

    print(resultado.get_outputs())

await main()

['La **IA generativa** es un área de la inteligencia artificial capaz de **generar contenido nuevo** a partir de patrones aprendidos durante su entrenamiento con grandes volúmenes de datos. Puede producir textos, imágenes, audio, vídeo, código y otros tipos de contenido.\n\nA diferencia de muchos sistemas de IA diseñados principalmente para clasificar información, detectar patrones o realizar predicciones, la IA generativa crea resultados adaptados a las instrucciones del usuario, conocidas como **prompts**. Sin embargo, esta distinción no es absoluta: algunos sistemas pueden combinar capacidades de análisis, predicción y generación.\n\n### ¿Cómo funciona?\n\nLa IA generativa utiliza modelos entrenados con grandes conjuntos de datos. Durante el entrenamiento, estos modelos aprenden relaciones, estructuras y regularidades presentes en la información. Después, cuando reciben una instrucción, generan una respuesta calculando qué elementos resultan más probables y adecuados para el context

In [7]:
# Otra manera de ejecutar el ejemplo anterior utilizando ·SequentialBuilder"
async def main():
    workflow = SequentialBuilder(
        participants=[redactor, revisor], # orden de ejecucion (solo enuncias los agentes directamente)
        chain_only_agent_responses=True # El revisor solo recibe el mensaje del nodo anterior (redactor)
    ).build()   
    resultado = await workflow.run("¿Qué es la IA generativa?", )

    salidas = resultado.get_outputs()  
    for salida in salidas:
        for mensaje in salida.messages:
            print(f"[{mensaje.author_name or 'assistant'}]")
            print(mensaje.text)
            print("-" * 50)    

await main()



[Revisor]
La **inteligencia artificial generativa** es una rama de la IA capaz de **generar contenido nuevo** a partir de patrones aprendidos durante su entrenamiento con grandes volúmenes de datos.

Puede producir, por ejemplo:

- **Texto**, como respuestas, resúmenes o historias.
- **Imágenes**, a partir de instrucciones escritas.
- **Audio y música**.
- **Vídeo**.
- **Código de programación**.

A diferencia de los sistemas diseñados principalmente para clasificar información, detectar patrones o seguir reglas predefinidas, la IA generativa crea resultados que no son una simple copia de un ejemplo concreto. Los produce combinando y transformando patrones aprendidos. Por ejemplo, un modelo de lenguaje puede recibir la instrucción *«explica la fotosíntesis»* y generar una explicación coherente.

Estos sistemas suelen basarse en modelos de aprendizaje automático, especialmente en **redes neuronales profundas**. En el caso de los modelos de lenguaje, el sistema genera una respuesta predi

In [8]:
# Se pueden observar los pasos intermedios en la ejecucion
async def main():
    workflow = SequentialBuilder(
        participants=[redactor, revisor], # orden de ejecucion (solo enuncias los agentes directamente)
        chain_only_agent_responses=True, # El revisor solo recibe el mensaje del nodo anterior (redactor)
        intermediate_output_from=[redactor]
    ).build()   
    resultado = await workflow.run("¿Qué es la IA generativa?", )

    print("=== BORRADOR DEL REDACTOR ===")
    for salida in resultado.get_intermediate_outputs():
        print(salida)

    print("\n=== RESPUESTA FINAL ===")
    for salida in resultado.get_outputs():
        print(salida)
 

await main()

=== BORRADOR DEL REDACTOR ===
La **IA generativa** es una rama de la inteligencia artificial capaz de **crear contenido nuevo** a partir de patrones aprendidos en grandes cantidades de datos.

Puede generar, entre otros:

- **Texto**, como respuestas, resúmenes o código.
- **Imágenes**, a partir de descripciones escritas.
- **Audio y música**.
- **Vídeos**.
- **Diseños y modelos tridimensionales**.

A diferencia de los sistemas que solo clasifican información o predicen un resultado, la IA generativa produce una respuesta o contenido original que mantiene características aprendidas de sus datos de entrenamiento. Por ejemplo, un modelo de lenguaje puede analizar una pregunta y generar una respuesta coherente; un modelo de imágenes puede crear una ilustración basándose en una instrucción.

Su funcionamiento suele basarse en modelos de aprendizaje automático, como las **redes neuronales profundas** y los **modelos fundacionales**. Estos modelos identifican relaciones y patrones en los dat

In [9]:
# Ejecucion de workflow en paralelo
# Agente 1: análisis técnico
tecnico = Agent(
    client=client,
    name="AnalistaTecnico",
    instructions=(
        "Analiza la viabilidad técnica de la propuesta. "
        "Considera arquitectura, integraciones y complejidad."
    ),
)

# Agente 2: análisis de riesgos
riesgos = Agent(
    client=client,
    name="AnalistaRiesgos",
    instructions=(
        "Identifica los principales riesgos de la propuesta, "
        "sus impactos y posibles medidas de mitigación."
    ),
)

# Agente 3: análisis de negocio
negocio = Agent(
    client=client,
    name="AnalistaNegocio",
    instructions=(
        "Evalúa el valor de negocio, los beneficios esperados "
        "y las limitaciones de la propuesta."
    ),
)

# Ejecutar los tres agentes en paralelo
workflow_paralelo = ConcurrentBuilder(
    participants=[tecnico, riesgos, negocio],
).build()

# Se ejecutan los tres agentes en paralelo, como son finales, se muestra la informacion de todos los agentes.
async def main():
    resultado = await workflow_paralelo.run(
        "Propuesta: implementar un asistente de IA que responda "
        "preguntas de los empleados utilizando documentación interna."
    )

    salidas = resultado.get_outputs()

    for salida in salidas:
        for mensaje in salida.messages:
            print(f"\n=== {mensaje.author_name} ===")
            print(mensaje.text)

await main()


=== AnalistaTecnico ===
## Evaluación general

La propuesta es **técnicamente viable** y relativamente madura si se implementa como un asistente basado en **RAG (Retrieval-Augmented Generation)**: el modelo no “aprende” permanentemente toda la documentación, sino que busca información relevante en las fuentes internas y genera una respuesta basada en esos fragmentos.

La complejidad principal no está en conectar un modelo de lenguaje, sino en:

- Mantener la documentación actualizada.
- Respetar permisos y confidencialidad.
- Integrar fuentes heterogéneas.
- Evitar respuestas inventadas o descontextualizadas.
- Medir la calidad y gestionar el ciclo de vida del sistema.

## Arquitectura propuesta

### 1. Interfaz de usuario

Canales posibles:

- Aplicación web interna.
- Microsoft Teams, Slack o Google Chat.
- Portal corporativo existente.
- API para integrarse con otros sistemas.

La interfaz debería mostrar:

- Respuesta generada.
- Fuentes utilizadas y enlaces al documento original.

In [10]:
# Agente que puede sintetizar toda la informacion de los tres agentes previos
sintetizador = Agent(
    client=client,
    name="Sintetizador",
    instructions=(
        "Recibirás análisis técnico, de riesgos y de negocio. "
        "Integra los tres en una evaluación ejecutiva. "
        "Distingue fortalezas, riesgos y recomendaciones. "
        "No inventes información que no aparezca en los análisis."
    ),
)

async def main():
    propuesta = (
        "Implementar un asistente de IA que responda preguntas "
        "de los empleados usando documentación interna."
    )

    # Fase 1: ejecución paralela y agregación
    resultado_paralelo = await workflow_paralelo.run(propuesta)

    salidas = resultado_paralelo.get_outputs()

    textos = []
    for salida in salidas:
        for mensaje in salida.messages:
            textos.append(
                f"{mensaje.author_name}:\n{mensaje.text}"
            )

    # Fase 2: sintetizar los resultados
    prompt_sintesis = (
        f"Propuesta original:\n{propuesta}\n\n"
        "Análisis de los especialistas:\n\n"
        + "\n\n".join(textos)
    )

    respuesta_final = await sintetizador.run(prompt_sintesis)

    print("=== EVALUACIÓN INTEGRADA ===")
    print(respuesta_final.text)

await main()


=== EVALUACIÓN INTEGRADA ===
# Evaluación ejecutiva

## Conclusión

La propuesta es **recomendable**, siempre que se implemente inicialmente como un **asistente documental de autoservicio, de solo lectura y bajo supervisión humana**. El mayor potencial está en reducir consultas repetitivas, acelerar la búsqueda de información y mejorar la autonomía de los empleados.

La viabilidad dependerá menos del modelo de IA que de tres factores críticos:

1. **Calidad, vigencia y consistencia de la documentación.**
2. **Aplicación estricta de los permisos de acceso.**
3. **Capacidad para limitar respuestas no sustentadas y derivar casos sensibles a personas responsables.**

Se recomienda avanzar mediante un piloto controlado antes de una expansión empresarial.

---

## Fortalezas

### Valor para el negocio

- Reduce el tiempo dedicado a localizar políticas, procedimientos y guías.
- Puede disminuir consultas repetitivas a RR. HH., TI, operaciones y soporte.
- Facilita la incorporación de nuevos e

In [11]:
# 1. Agentes especialistas
tecnico = Agent(
    client=client,
    name="AnalistaTecnico",
    instructions="Evalúa la viabilidad técnica de la propuesta.",
)

riesgos = Agent(
    client=client,
    name="AnalistaRiesgos",
    instructions="Identifica riesgos técnicos, operativos y de seguridad.",
)

negocio = Agent(
    client=client,
    name="AnalistaNegocio",
    instructions="Evalúa el valor de negocio, beneficios y costes.",
)


# 2. Agente que sintetiza los resultados
sintetizador = Agent(
    client=client,
    name="Sintetizador",
    instructions=(
        "Integra los análisis técnico, de riesgos y de negocio "
        "en un informe ejecutivo. Incluye viabilidad, riesgos, "
        "beneficios y una recomendación final."
    ),
)


# 3. Agregador personalizado: forma parte del workflow
async def sintetizar_resultados(resultados):
    informe = "\n\n".join(
        f"Análisis {i + 1}:\n{r.agent_response.text}"
        for i, r in enumerate(resultados)
    )

    respuesta = await sintetizador.run(
        f"Elabora un informe ejecutivo integrado a partir "
        f"de los siguientes análisis:\n\n{informe}"
    )

    return respuesta.text


# 4. Construcción completa del workflow
workflow = (
    ConcurrentBuilder(
        participants=[tecnico, riesgos, negocio]
    )
    .with_aggregator(sintetizar_resultados)
    .build()
)


# 5. Ejecutar el workflow
async def main():
    resultado = await workflow.run(
        "Evaluar una solución de IA que responda preguntas "
        "de empleados utilizando documentación interna."
    )

    for salida in resultado.get_outputs():
        print(salida)


await main()

# Informe ejecutivo integrado  
## Asistente de IA para responder consultas de empleados con documentación interna

### 1. Resumen ejecutivo

La solución propuesta es **técnicamente viable y potencialmente atractiva desde el punto de vista de negocio**, siempre que se implemente como un asistente documental basado en fuentes verificables, con control estricto de permisos y supervisión humana para asuntos sensibles.

El enfoque recomendado es una arquitectura **RAG (Retrieval-Augmented Generation)**: el sistema localiza información relevante en repositorios corporativos autorizados y genera respuestas fundamentadas en esos documentos, mostrando las fuentes utilizadas.

El principal valor esperado es la reducción de consultas repetitivas a RR. HH., TI, soporte y otras áreas internas, así como una mejora en la rapidez del autoservicio, la incorporación de empleados y la consistencia de las respuestas. Sin embargo, los beneficios dependen directamente de la calidad y gobierno de la documen

In [12]:
# Ejecución condicional. En este caso se ejecutan nodos en caso se cumpla una condicion
# Como hay circularidad, se debe poner un maximo de profundidad
MAX_REVISIONES = 3


# 1. Estructura de la decisión
class Decision(BaseModel):
    texto: str
    aprobado: bool
    motivo: str
    intento: int = 0


# 2. Agente evaluador
evaluador = Agent(
    client=client,
    name="Evaluador",
    instructions=(
        "Evalúa si el texto es claro, correcto y completo. "
        "Comprueba la gramática y la precisión conceptual. "
        "Devuelve aprobado=True si cumple los criterios; "
        "en caso contrario, aprobado=False. "
        "Explica brevemente el motivo."
    ),
    default_options={"response_format": Decision},
)


# 3. Agente revisor
revisor = Agent(
    client=client,
    name="Revisor",
    instructions=(
        "Corrige el texto recibido teniendo en cuenta "
        "las observaciones del evaluador. "
        "Conserva su significado y devuelve solamente "
        "la versión corregida, sin explicaciones."
    ),
)


# 4. Ejecutor inicial: convierte la entrada en una Decision
class Inicio(Executor):
    @handler
    async def iniciar(self,texto: str,ctx: WorkflowContext[Decision]) -> None:
        decision = Decision(
            texto=texto,
            aprobado=False,
            motivo="Pendiente de evaluación",
            intento=0,
        )
        await ctx.send_message(decision)


# 5. Ejecutor evaluador
class AnalizarDecision(Executor):
    def __init__(self, agente: Agent, id: str):
        super().__init__(id=id)
        self.agente = agente

    @handler
    async def analizar(self,decision: Decision,ctx: WorkflowContext[Decision]) -> None:
        
        respuesta = await self.agente.run(f"Evalúa el siguiente texto:\n\n{decision.texto}")
        evaluacion = Decision.model_validate_json(respuesta.text)

        # Preservar el texto y el contador de revisiones
        evaluacion.texto = decision.texto
        evaluacion.intento = decision.intento

        print(
            f"Evaluación {decision.intento}: "
            f"{'APROBADO' if evaluacion.aprobado else 'NO APROBADO'}"
        )
        print(f"Motivo: {evaluacion.motivo}")

        await ctx.send_message(evaluacion)


# 6. Ejecutor revisor: corrige y devuelve el texto al evaluador
class Revisar(Executor):
    def __init__(self, agente: Agent, id: str):
        super().__init__(id=id)
        self.agente = agente

    @handler
    async def revisar(self,decision: Decision,ctx: WorkflowContext[Decision]) -> None:
        respuesta = await self.agente.run(
            f"Motivo de revisión: {decision.motivo}\n\n"
            f"Texto que debes corregir:\n{decision.texto}"
        )

        decision_corregida = Decision(
            texto=respuesta.text,
            aprobado=False,
            motivo="Pendiente de nueva evaluación",
            intento=decision.intento + 1,
        )

        print(f"Enviando texto corregido. Revisión {decision_corregida.intento}")

        # No termina el workflow: vuelve al evaluador
        await ctx.send_message(decision_corregida)


# 7. Ejecutor terminal: publica el texto aprobado
class Publicar(Executor):
    @handler
    async def publicar(self,decision: Decision,ctx: WorkflowContext[Never, str],) -> None:
        await ctx.yield_output(
            f"INFORME APROBADO\n\n"
            f"{decision.texto}\n\n"
            f"Motivo: {decision.motivo}\n"
            f"Revisiones realizadas: {decision.intento}"
        )


# 8. Ejecutor terminal si se alcanza el máximo de revisiones
class RevisionManual(Executor):
    @handler
    async def escalar(self,decision: Decision,ctx: WorkflowContext[Never, str]) -> None:
        await ctx.yield_output(
            f"REQUIERE REVISIÓN MANUAL\n\n"
            f"Texto actual:\n{decision.texto}\n\n"
            f"Última observación: {decision.motivo}\n"
            f"Revisiones realizadas: {decision.intento}"
        )


# 9. Crear los ejecutores
inicio = Inicio(id="inicio")
analizador = AnalizarDecision(evaluador, id="analizador")
revisar = Revisar(revisor, id="revisar")
publicar = Publicar(id="publicar")
revision_manual = RevisionManual(id="revision_manual")


# 10. Construir el workflow con rutas condicionales y ciclo
workflow = (
    WorkflowBuilder(start_executor=inicio)
    .add_edge(inicio, analizador)
    .add_edge(
        analizador,
        publicar,
        condition=lambda d: d.aprobado,
    )
    .add_edge(
        analizador,
        revisar,
        condition=lambda d: (not d.aprobado and d.intento < MAX_REVISIONES),
    )
    .add_edge(
        analizador,
        revision_manual,
        condition=lambda d: (not d.aprobado and d.intento >= MAX_REVISIONES),
    )
    .add_edge(revisar, analizador)
    .build()
)


# 11. Ejecutar todo el workflow
async def main():
    # Se coloca texto con error de concordancia
    texto_inicial = ("La inteligencia artificial son una tecnología que ayuda.")
    resultado = await workflow.run(texto_inicial)

    print("\n=== RESULTADO FINAL ===")
    for salida in resultado.get_outputs():
        print(salida)

await main()

Evaluación 0: NO APROBADO
Motivo: Presenta un error de concordancia: debe decir «La inteligencia artificial es una tecnología…». Además, la afirmación es demasiado vaga y no explica cómo ayuda.
Enviando texto corregido. Revisión 1
Evaluación 1: APROBADO
Motivo: El texto es claro, gramaticalmente correcto y conceptualmente preciso, aunque ofrece una definición general y breve.

=== RESULTADO FINAL ===
INFORME APROBADO

La inteligencia artificial es una tecnología que ayuda a automatizar tareas y analizar grandes cantidades de datos.

Motivo: El texto es claro, gramaticalmente correcto y conceptualmente preciso, aunque ofrece una definición general y breve.
Revisiones realizadas: 1
